In [1]:
import requests
import pandas as pd
import numpy as np
import time
from pathlib import Path

# Configuración
PARTIDOS = {
    "General Arenales": {"lat": -34.31, "lon": -61.10},
    "Leandro N. Alem":  {"lat": -34.52, "lon": -61.38},
    "Junín":            {"lat": -34.59, "lon": -60.95},
    "Lincoln":          {"lat": -34.87, "lon": -61.53},
    "General Pinto":    {"lat": -34.76, "lon": -61.89},
}

PARAMS = "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,T2MDEW"
BASE_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"

# Carpeta de salida (sube dos niveles desde notebooks/ hasta rinde-soja/)
OUTPUT_DIR = Path("..") / "data" / "raw" / "nasa_power"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # Crea la carpeta si no existe

all_data = []

for partido, coords in PARTIDOS.items():
    print(f"Descargando {partido}...", end=" ", flush=True)
    
    url = (
        f"{BASE_URL}?"
        f"parameters={PARAMS}"
        f"&community=AG"
        f"&longitude={coords['lon']}"
        f"&latitude={coords['lat']}"
        f"&start=19990901"
        f"&end=20250430"
        f"&format=JSON"
    )
    
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    
    df = pd.DataFrame(data["properties"]["parameter"])
    df.index.name = "fecha"
    df = df.reset_index().rename(columns={"index": "fecha"})
    df["partido"] = partido
    df["lat"] = coords["lat"]
    df["lon"] = coords["lon"]
    all_data.append(df)
    print(f"OK — {len(df)} días")
    time.sleep(2)

# Unir y limpiar
df_clima = pd.concat(all_data, ignore_index=True)
df_clima["fecha"] = pd.to_datetime(df_clima["fecha"], format="%Y%m%d")

# -999 = missing de NASA POWER → reemplazar con NaN
numeric_cols = ["T2M","T2M_MAX","T2M_MIN","PRECTOTCORR","ALLSKY_SFC_SW_DWN","RH2M","T2MDEW"]
for col in numeric_cols:
    df_clima[col] = df_clima[col].replace(-999, np.nan)

# Guardar
df_clima.to_csv(OUTPUT_DIR / "nasa_power_zona_coop_1999_2025.csv", index=False)

print(f"\nListo! Shape: {df_clima.shape}")
print(f"Período: {df_clima['fecha'].min().date()} a {df_clima['fecha'].max().date()}")
print(f"Guardado en: {OUTPUT_DIR / 'nasa_power_zona_coop_1999_2025.csv'}")

Descargando General Arenales... OK — 9374 días
Descargando Leandro N. Alem... OK — 9374 días
Descargando Junín... OK — 9374 días
Descargando Lincoln... OK — 9374 días
Descargando General Pinto... OK — 9374 días

Listo! Shape: (46870, 11)
Período: 1999-09-01 a 2025-04-30
Guardado en: ..\data\raw\nasa_power\nasa_power_zona_coop_1999_2025.csv


In [2]:
import os
print("El notebook está corriendo desde:", os.getcwd())
print("\nEl CSV se guardó en:")
print(os.path.abspath(str(OUTPUT_DIR / "nasa_power_zona_coop_1999_2025.csv")))

El notebook está corriendo desde: C:\Users\agust\Python

El CSV se guardó en:
C:\Users\agust\data\raw\nasa_power\nasa_power_zona_coop_1999_2025.csv
